# LLM Faithfulness — Results Analysis

Loads `.jsonl` result files from Drive and computes
all metrics, plots, and statistical tests.

**Contents:**
1. Setup & load results
2. Data quality check
3. Faithfulness metrics (all tasks)
4. Visualisations per experiment
5. BERTScore — reasoning consistency (H3)
6. Summary table (all primary metrics)
7. H2 statistical comparison — McNemar's test
8. Confidence shift analysis
9. Individual result inspection


## 1. Setup


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# CONFIG

# Must match DRIVE_RESULTS in thesis_experiments.ipynb
DRIVE_RESULTS = "/content/drive/MyDrive/thesis_results"

# Repo location
REPO_DIR = "/content/thesis"
REPO_URL = "https://github.com/matiimonti/llm-faithfulness-financial-sentiment-analysis.git"

# Imports
import os, sys, json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

plt.rcParams.update({'figure.dpi': 130, 'font.size': 11})
sns.set_theme(style='whitegrid')

# Clone repo if needed (for metrics/ modules)
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

In [ ]:
# Analysis deps only 
!pip install -q statsmodels bert-score

## 2. Load results


In [ ]:
import glob

def load_task(task_prefix: str) -> pd.DataFrame:
    """Load all per-model files for a task and merge into one DataFrame.
    Falls back to the legacy single-file format if no per-model files exist.
    """
    pattern = os.path.join(DRIVE_RESULTS, f"{task_prefix}_*.jsonl")
    files = sorted(glob.glob(pattern))

    # Fallback to legacy single file
    if not files:
        legacy = os.path.join(DRIVE_RESULTS, f"{task_prefix}.jsonl")
        if os.path.exists(legacy):
            files = [legacy]

    if not files:
        print(f"[not found] {task_prefix}_*.jsonl")
        return pd.DataFrame()

    rows = []
    for path in files:
        rows.extend(json.loads(l) for l in open(path) if l.strip())

    if not rows:
        return pd.DataFrame()

    df = pd.DataFrame(rows)
    # Flatten the 'extra' dict column into top-level columns
    if 'extra' in df.columns:
        extra = pd.json_normalize(df['extra'])
        df = pd.concat([df.drop(columns='extra'), extra], axis=1)

    fnames = [os.path.basename(f) for f in files]
    models = sorted(df['model'].unique()) if 'model' in df.columns else []
    print(f"{task_prefix:20s}  {len(df):>5} rows files: {fnames} models: {models}")
    return df


baseline = load_task('baseline')
redaction = load_task('redaction')
counterfact = load_task('counterfactual')
cot = load_task('cot_intervention')
stability = load_task('stability')

## 3. Data quality check


In [ ]:
def quality_report(df: pd.DataFrame, task: str) -> None:
    if df.empty:
        print(f"{task}: no data\n")
        return
    print(f"── {task.upper()} ──")
    for model, g in df.groupby('model'):
        n = len(g)
        label_dist = g['label'].value_counts().to_dict() if 'label' in g.columns else {}

        if task == 'baseline':
            parse_fail = g['correct'].isna().sum()
            print(f"{model}: n={n}  parse_failures={parse_fail}  labels={label_dist}")

        elif task == 'redaction':
            attr_ok = (g.get('attribution_status', pd.Series()) == 'ok').sum()
            parse_fail = (g.get('attribution_status', pd.Series()) == 'parse_failed').sum()
            empty = (g.get('attribution_status', pd.Series()) == 'empty_phrases').sum()
            print(f"{model}: n={n}  attr_ok={attr_ok}  parse_failed={parse_fail}  empty={empty}")

        elif task == 'counterfactual':
            valid = g.get('finbert_valid', pd.Series(dtype=bool)).sum()
            print(f"{model}: n={n}  finbert_valid={valid} ({valid/n:.1%})")

        elif task == 'cot_intervention':
            scored = g['faithful_followed_cot'].notna().sum()
            skipped = n - scored
            print(f"{model}: n={n}  scored={scored}  skipped_empty_reasoning={skipped}")

        elif task == 'stability':
            v1_fail = g['predict_v1'].isna().sum() if 'predict_v1' in g.columns else '—'
            v2_fail = g['predict_v2'].isna().sum() if 'predict_v2' in g.columns else '—'
            v3_fail = g['predict_v3'].isna().sum() if 'predict_v3' in g.columns else '—'
            print(f"{model}: n={n}  parse_failures v1={v1_fail} v2={v2_fail} v3={v3_fail}")
    print()


quality_report(baseline, 'baseline')
quality_report(redaction, 'redaction')
quality_report(counterfact, 'counterfactual')
quality_report(cot, 'cot_intervention')
quality_report(stability, 'stability')

## 4. Faithfulness metrics — all tasks


In [ ]:
from metrics.faithfulness import run as compute_faithfulness

all_metrics = compute_faithfulness(results_dir=DRIVE_RESULTS)

## 5. Visualisations


In [ ]:
# Helpers

# Short display names for models
MODEL_LABELS = {
    'meta-llama/Llama-3.2-3B-Instruct': 'Llama-3.2-3B',
    'google/gemma-2-2b-it': 'Gemma-2-2B',
    'FinGPT/fingpt-sentiment_llama2-7b-lora': 'FinGPT-7B',
}

def short(model_name: str) -> str:
    return MODEL_LABELS.get(model_name, model_name.split('/')[-1])

COLORS = sns.color_palette('Set2', 3)

In [ ]:
# 5a. Baseline accuracy

if not baseline.empty:
    acc = (
        baseline.groupby('model')
        .apply(lambda g: g['correct'].dropna().mean())
        .rename('accuracy')
        .reset_index()
    )
    acc['model_short'] = acc['model'].map(short)

    fig, ax = plt.subplots(figsize=(6, 3.5))
    bars = ax.barh(acc['model_short'], acc['accuracy'], color=COLORS[:len(acc)])
    ax.bar_label(bars, fmt='%.3f', padding=4)
    ax.set_xlim(0, 1.05)
    ax.set_xlabel('Accuracy')
    ax.set_title('Baseline Classification Accuracy\n(Financial PhraseBank — sentences_allagree)')
    plt.tight_layout()
    plt.savefig(os.path.join(DRIVE_RESULTS, 'plot_baseline_accuracy.pdf'), bbox_inches='tight')
    plt.show()

In [ ]:
# 5b. Redaction — faithful_destabilised_rate

if not redaction.empty:
    # Compute per model
    scored = redaction[redaction['faithful_destabilised'].notna()]
    rates = (
        scored.groupby('model')
        .agg(
            destabilised=('faithful_destabilised', 'mean'),
            unknown=('faithful_unknown', 'mean'),
            flip=('faithful_flip', 'mean'),
        )
        .reset_index()
    )
    rates['model_short'] = rates['model'].map(short)

    x = np.arange(len(rates))
    w = 0.25
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar(x - w, rates['destabilised'], w, label='Destabilised (primary)', color=COLORS[0])
    ax.bar(x,rates['unknown'], w, label='Unknown only', color=COLORS[1])
    ax.bar(x + w, rates['flip'], w, label='Any flip', color=COLORS[2])
    ax.set_xticks(x)
    ax.set_xticklabels(rates['model_short'])
    ax.set_ylabel('Rate')
    ax.set_ylim(0, 1.05)
    ax.set_title('Redaction — Faithfulness Signals\n(H1: cited phrases are load-bearing)')
    ax.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(DRIVE_RESULTS, 'plot_redaction.pdf'), bbox_inches='tight')
    plt.show()

In [ ]:
# 5c. Counterfactual

if not counterfact.empty:
    def cf_rates(g):
        n = len(g)
        valid = g[g['finbert_valid'] == True]
        return pd.Series({
            'faithful_correct': g['faithful'].mean(),
            'faithful_correct_valid': valid['faithful'].mean() if len(valid) else np.nan,
            'faithful_flip': g['faithful_flip'].mean() if 'faithful_flip' in g.columns else np.nan,
            'finbert_valid_rate': len(valid) / n,
        })

    cf = counterfact.groupby('model').apply(cf_rates).reset_index()
    cf['model_short'] = cf['model'].map(short)

    x = np.arange(len(cf))
    w = 0.28
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar(x - w, cf['faithful_correct'], w, label='Faithful correct (primary)', color=COLORS[0])
    ax.bar(x, cf['faithful_correct_valid'],  w, label='Faithful correct | FinBERT valid', color=COLORS[1])
    ax.bar(x + w, cf['faithful_flip'], w, label='Any flip', color=COLORS[2])
    ax.set_xticks(x)
    ax.set_xticklabels(cf['model_short'])
    ax.set_ylabel('Rate')
    ax.set_ylim(0, 1.05)
    ax.set_title('Counterfactual — Faithfulness Signals\n(H1: model tracks flipped sentiment signal)')
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.savefig(os.path.join(DRIVE_RESULTS, 'plot_counterfactual.pdf'), bbox_inches='tight')
    plt.show()

In [ ]:
# 5d. CoT Intervention

if not cot.empty:
    scored_cot = cot[cot['faithful_followed_cot'].notna()]
    cot_rates = (
        scored_cot.groupby('model')
        .agg(
            followed_cot=('faithful_followed_cot', 'mean'),
            robust=('faithful_robust', 'mean'),
        )
        .reset_index()
    )
    cot_rates['model_short'] = cot_rates['model'].map(short)

    x = np.arange(len(cot_rates))
    w = 0.32
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar(x - w/2, cot_rates['followed_cot'], w, label='Followed CoT (causally driven)', color=COLORS[0])
    ax.bar(x + w/2, cot_rates['robust'], w, label='Robust / post-hoc', color=COLORS[1])
    ax.set_xticks(x)
    ax.set_xticklabels(cot_rates['model_short'])
    ax.set_ylabel('Rate')
    ax.set_ylim(0, 1.05)
    ax.set_title('CoT Intervention — Faithfulness Signals\n(Novel contribution: is CoT causal or post-hoc?)')
    ax.legend()
    # Reference line at 0.5
    ax.axhline(0.5, color='grey', linestyle='--', linewidth=0.8, label='chance')
    plt.tight_layout()
    plt.savefig(os.path.join(DRIVE_RESULTS, 'plot_cot_intervention.pdf'), bbox_inches='tight')
    plt.show()

In [ ]:
# 5e. Prompt Stability — label agreement

if not stability.empty:
    stab_rates = (
        stability.groupby('model')
        .agg(
            all_agree=('all_agree', 'mean'),
            v1_v2=('predict_agreement_v1_v2', 'mean'),
            v1_v3=('predict_agreement_v1_v3', 'mean'),
            v2_v3=('predict_agreement_v2_v3', 'mean'),
        )
        .reset_index()
    )
    stab_rates['model_short'] = stab_rates['model'].map(short)

    x = np.arange(len(stab_rates))
    w = 0.18
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(x - 1.5*w, stab_rates['all_agree'], w, label='All 3 agree (primary)', color=COLORS[0])
    ax.bar(x - 0.5*w, stab_rates['v1_v2'], w, label='v1 vs v2', color=COLORS[1])
    ax.bar(x + 0.5*w, stab_rates['v1_v3'], w, label='v1 vs v3', color=COLORS[2])
    ax.bar(x + 1.5*w, stab_rates['v2_v3'], w, label='v2 vs v3', color='steelblue')
    ax.set_xticks(x)
    ax.set_xticklabels(stab_rates['model_short'])
    ax.set_ylabel('Agreement rate')
    ax.set_ylim(0, 1.05)
    ax.set_title('Prompt Stability — Label Agreement (H3)')
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.savefig(os.path.join(DRIVE_RESULTS, 'plot_stability_agreement.pdf'), bbox_inches='tight')
    plt.show()

## 6. BERTScore — reasoning consistency (H3)


In [ ]:
from metrics.similarity import compute_bertscore

bertscore_metrics = compute_bertscore(results_dir=DRIVE_RESULTS)

In [ ]:
# Plot BERTScore F1 per model and paraphrase pair

if bertscore_metrics:
    bs_rows = []
    for model, m in bertscore_metrics.items():
        for pair in ('v1_v2', 'v1_v3', 'v2_v3'):
            v = m.get(f'bertscore_f1_{pair}')
            if v is not None:
                bs_rows.append({'model': short(model), 'pair': pair, 'f1': v})
    bs_df = pd.DataFrame(bs_rows)

    fig, ax = plt.subplots(figsize=(7, 4))
    sns.barplot(data=bs_df, x='model', y='f1', hue='pair', palette='Set2', ax=ax)
    ax.set_ylabel('BERTScore F1')
    ax.set_ylim(0, 1.0)
    ax.set_title('Prompt Stability — Reasoning Consistency (H3)\nBERTScore F1 across paraphrase pairs')
    ax.legend(title='Pair')
    plt.tight_layout()
    plt.savefig(os.path.join(DRIVE_RESULTS, 'plot_bertscore.pdf'), bbox_inches='tight')
    plt.show()

## 7. Summary table — all primary metrics



In [ ]:
rows = []

# Collect all models seen across any experiment
all_models = set()
for df in [baseline, redaction, counterfact, cot, stability]:
    if not df.empty and 'model' in df.columns:
        all_models.update(df['model'].unique())

for model in sorted(all_models):
    row = {'Model': short(model)}

    # Baseline accuracy
    if not baseline.empty:
        g = baseline[baseline['model'] == model]
        row['Accuracy'] = g['correct'].dropna().mean() if len(g) else np.nan

    # Redaction: faithful_destabilised_rate
    if not redaction.empty:
        g = redaction[(redaction['model'] == model) & redaction['faithful_destabilised'].notna()]
        row['Redaction\nDestabilised'] = g['faithful_destabilised'].mean() if len(g) else np.nan

    # Counterfactual: faithful_correct_among_valid
    if not counterfact.empty:
        g = counterfact[(counterfact['model'] == model) & (counterfact['finbert_valid'] == True)]
        row['Counterfactual\nCorrect|Valid'] = g['faithful'].mean() if len(g) else np.nan

    # CoT intervention: followed_cot and robust
    if not cot.empty:
        g = cot[(cot['model'] == model) & cot['faithful_followed_cot'].notna()]
        row['CoT\nFollowed'] = g['faithful_followed_cot'].mean() if len(g) else np.nan
        row['CoT\nRobust']   = g['faithful_robust'].mean() if len(g) else np.nan

    # Stability: all_agree_rate
    if not stability.empty:
        g = stability[stability['model'] == model]
        row['Stability\nAll Agree'] = g['all_agree'].mean() if len(g) else np.nan

    # BERTScore mean
    if bertscore_metrics and model in bertscore_metrics:
        row['BERTScore\nMean F1'] = bertscore_metrics[model].get('bertscore_f1_mean')

    rows.append(row)

summary = pd.DataFrame(rows).set_index('Model')
print(summary.to_string(float_format='{:.3f}'.format))
summary

In [ ]:
# Heatmap of the summary table

fig, ax = plt.subplots(figsize=(10, max(2.5, len(summary) * 1.2)))
sns.heatmap(
    summary.astype(float),
    annot=True, fmt='.3f',
    cmap='YlOrRd', vmin=0, vmax=1,
    linewidths=0.5, ax=ax
)
ax.set_title('Primary Faithfulness Metrics — All Models')
ax.set_ylabel('')
plt.tight_layout()
plt.savefig(os.path.join(DRIVE_RESULTS, 'plot_summary_heatmap.pdf'), bbox_inches='tight')
plt.show()

## 8. H2 Statistical comparison — McNemar's test

Tests whether two models differ significantly on the same faithfulness binary
outcome (faithful = True/False) for the same observations.

McNemar's test is appropriate here because observations are paired
(same sentence, same task, different model).

Null hypothesis: no difference in the proportion of faithful predictions
between model A and model B.


In [ ]:
from statsmodels.stats.contingency_tables import mcnemar
from itertools import combinations

def run_mcnemar(df: pd.DataFrame, outcome_col: str, task_name: str) -> None:
    """Run McNemar's test on all model pairs for a binary faithfulness column."""
    if df.empty or outcome_col not in df.columns:
        return

    models = sorted(df['model'].unique())
    if len(models) < 2:
        print(f"{task_name}: only one model — skipping\n")
        return

    print(f"── {task_name.upper()} ({outcome_col}) ──")

    for m_a, m_b in combinations(models, 2):
        # Align on observation ID — inner join to get paired observations only
        a = df[df['model'] == m_a][['id', outcome_col]].dropna().set_index('id')[outcome_col].astype(bool)
        b = df[df['model'] == m_b][['id', outcome_col]].dropna().set_index('id')[outcome_col].astype(bool)
        paired = pd.DataFrame({'a': a, 'b': b}).dropna()

        if len(paired) < 10:
            print(f"  {short(m_a)} vs {short(m_b)}: too few paired observations ({len(paired)}) — skipping")
            continue

        # 2×2 contingency table: rows = model A, cols = model B
        table = pd.crosstab(paired['a'], paired['b']).values
        if table.shape != (2, 2):
            print(f"  {short(m_a)} vs {short(m_b)}: degenerate table — skipping")
            continue

        result = mcnemar(table, exact=True)
        sig = '***' if result.pvalue < 0.001 else '**' if result.pvalue < 0.01 else '*' if result.pvalue < 0.05 else 'ns'
        rate_a = paired['a'].mean()
        rate_b = paired['b'].mean()
        print(
            f"  {short(m_a)} ({rate_a:.3f}) vs {short(m_b)} ({rate_b:.3f}): "
            f"p={result.pvalue:.4f} {sig}  (n={len(paired)} paired)"
        )
    print()


# Run McNemar's for each task's primary outcome
run_mcnemar(redaction, 'faithful_destabilised', 'Redaction')
run_mcnemar(counterfact, 'faithful', 'Counterfactual')
run_mcnemar(cot, 'faithful_followed_cot', 'CoT Intervention')
run_mcnemar(stability, 'all_agree', 'Prompt Stability')

## 9. Confidence shift analysis

How much does confidence drop after each perturbation, per model?
Positive shift = confidence fell after the perturbation.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=False)
tasks_conf = [
    (redaction, 'confidence_shift', 'Redaction'),
    (counterfact, 'confidence_shift', 'Counterfactual'),
    (cot, 'confidence_shift', 'CoT Intervention'),
]

for ax, (df, col, title) in zip(axes, tasks_conf):
    if df.empty or col not in df.columns:
        ax.set_visible(False)
        continue
    plot_data = df[['model', col]].dropna()
    plot_data['model_short'] = plot_data['model'].map(short)
    sns.boxplot(data=plot_data, x='model_short', y=col, palette='Set2', ax=ax)
    ax.axhline(0, color='grey', linestyle='--', linewidth=0.8)
    ax.set_title(title)
    ax.set_xlabel('')
    ax.set_ylabel('Confidence shift (original − perturbed)')

plt.suptitle('Confidence Shift After Perturbation (positive = dropped)', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(DRIVE_RESULTS, 'plot_confidence_shift.pdf'), bbox_inches='tight')
plt.show()

## 10. Individual result inspection

Browse specific observations — useful for qualitative analysis and
identifying failure modes to discuss in the thesis.


In [ ]:
# Interesting CoT examples: model followed injected counter-reasoning

if not cot.empty:
    followed = cot[cot['faithful_followed_cot'] == True].head(3)
    for _, r in followed.iterrows():
        print(f"ID {r['id']} | Model: {short(r['model'])} | Label: {r['label']}")
        print(f"Original text: {r['text'][:120]}...")
        print(f"Original predict: {r['predict']}")
        print(f"Counter-reasoning: {str(r.get('counter_reasoning',''))[:200]}...")
        print(f"Intervened predict: {r['explain_predict']}")
        print()

In [ ]:
# Redaction: cases where attribution succeeded but prediction didn't change

if not redaction.empty:
    not_faithful = redaction[
        (redaction['attribution_status'] == 'ok') &
        (redaction['faithful_destabilised'] == False)
    ].head(3)
    for _, r in not_faithful.iterrows():
        print(f"ID {r['id']} | Model: {short(r['model'])} | Label: {r['label']}")
        print(f"Original text: {r['text'][:120]}...")
        print(f"Key phrases: {r.get('key_phrases', [])}")
        print(f"Redacted text: {str(r.get('explain',''))[:120]}...")
        print(f"Original pred: {r['predict']} -> Redacted pred: {r['explain_predict']}")
        print()

In [ ]:
# ── Stability: cases where labels disagreed across paraphrases ───────────────

if not stability.empty:
    disagreed = stability[stability['all_agree'] == False].head(3)
    for _, r in disagreed.iterrows():
        print(f"ID {r['id']} | Model: {short(r['model'])} | Label: {r['label']}")
        print(f"Text: {r['text'][:120]}...")
        print(f"v1 -> {r.get('predict_v1')}  |  v2 -> {r.get('predict_v2')}  |  v3 -> {r.get('predict_v3')}")
        print()